# 01 - Data Cleaning: Food Delivery Analysis

**Dataset:** Food Delivery Time dataset (`train.csv`, 45,593 orders, 20 columns)
**Goal:** Clean the raw file, create new features, and save an analysis-ready file to `data/clean/`.

> `test.csv` has no `Time_taken(min)` column and `Sample_Submission.csv` is only a Kaggle submission template, so this project uses **train.csv only**.

## Data dictionary

| Column | Meaning |
|---|---|
| ID | Unique order ID |
| Delivery_person_ID | Rider ID (starts with a city code, e.g. `BANG`, `PUNE`) |
| Delivery_person_Age / Ratings | Rider age and average rating |
| Restaurant_latitude / longitude | Pickup location |
| Delivery_location_latitude / longitude | Drop location |
| Order_Date | Date of the order (dd-mm-yyyy) |
| Time_Orderd / Time_Order_picked | Time order was placed / picked up |
| Weatherconditions | Weather at delivery time |
| Road_traffic_density | Low, Medium, High, Jam |
| Vehicle_condition | Vehicle condition score (0 to 3) |
| Type_of_order | Snack, Meal, Drinks, Buffet |
| Type_of_vehicle | motorcycle, scooter, electric_scooter, bicycle |
| multiple_deliveries | Other orders the rider carried at the same time |
| Festival | Was it a festival day (Yes/No) |
| City | City type: Metropolitian, Urban, Semi-Urban |
| Time_taken(min) | **Target:** delivery time in minutes (stored as text like `(min) 24`) |

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/train.csv")
print(df.shape)
df.head()

## 1. First inspection

In [ ]:
df.info()
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs :", df['ID'].duplicated().sum())
print("Real NaN values:", df.isna().sum().sum())

**Observation:** `isna()` finds nothing, but several columns contain the text `"NaN "` (with a trailing space). Pandas treats it as normal text, so the missing values are hidden. Let's count them.

In [ ]:
hidden = {}
for c in df.select_dtypes(include=['object', 'string']).columns:
    n = (df[c].astype(str).str.strip() == 'NaN').sum()
    if n:
        hidden[c] = n
pd.Series(hidden, name="hidden NaN count")

## 2. Clean text and missing markers

In [ ]:
text_cols = df.select_dtypes(include=['object', 'string']).columns
for c in text_cols:
    df[c] = df[c].astype(str).str.strip().replace({'NaN': np.nan, 'nan': np.nan})

df.isna().sum()[df.isna().sum() > 0]

## 3. Rename columns and fix data types

In [ ]:
df = df.rename(columns={'Time_Orderd': 'time_ordered', 'Time_taken(min)': 'time_taken_min'})
df.columns = df.columns.str.lower()

df['delivery_person_age']     = pd.to_numeric(df['delivery_person_age'])
df['delivery_person_ratings'] = pd.to_numeric(df['delivery_person_ratings'])
df['multiple_deliveries']     = pd.to_numeric(df['multiple_deliveries'])

# '(min) 24' -> 24
df['time_taken_min'] = df['time_taken_min'].str.extract(r'(\d+)').astype(int)

# 'conditions Sunny' -> 'Sunny'
df['weatherconditions'] = df['weatherconditions'].str.replace('conditions ', '', regex=False)

df['order_date'] = pd.to_datetime(df['order_date'], format='%d-%m-%Y')
df.dtypes

## 4. Fix invalid values

Checks on the raw data showed:
- Some **ratings are 6.0** (the scale ends at 5)
- Some **riders are aged 15** (below the legal working age)
- **Restaurant coordinates are negative** in some rows (India has positive lat/long)
- **3,640 rows have restaurant lat/long = 0**, which is a missing value, not a real location

In [ ]:
print("Ratings > 5 :", (df['delivery_person_ratings'] > 5).sum())
print("Age < 18    :", (df['delivery_person_age'] < 18).sum())
print("Negative restaurant lat/long:", (df['restaurant_latitude'] < 0).sum())
print("Zero restaurant coordinates :", ((df['restaurant_latitude'] == 0) & (df['restaurant_longitude'] == 0)).sum())

df.loc[df['delivery_person_ratings'] > 5, 'delivery_person_ratings'] = np.nan
df.loc[df['delivery_person_age'] < 18, 'delivery_person_age'] = np.nan

# negative coordinates are sign errors, so take the absolute value
df['restaurant_latitude']  = df['restaurant_latitude'].abs()
df['restaurant_longitude'] = df['restaurant_longitude'].abs()

# zero coordinates = missing
zero = (df['restaurant_latitude'] == 0) & (df['restaurant_longitude'] == 0)
df.loc[zero, ['restaurant_latitude', 'restaurant_longitude']] = np.nan

## 5. Feature engineering

**Distance** is the most important new column. It is calculated from the coordinates with the Haversine formula (straight-line km between pickup and drop).

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine(df['restaurant_latitude'], df['restaurant_longitude'],
                              df['delivery_location_latitude'], df['delivery_location_longitude'])
df['distance_km'].describe().round(2)

In [ ]:
# Time features. If time_ordered is missing, use the pickup time (it is only minutes later)
ordered_hour = pd.to_datetime(df['time_ordered'],      format='%H:%M:%S', errors='coerce').dt.hour
picked_hour  = pd.to_datetime(df['time_order_picked'], format='%H:%M:%S', errors='coerce').dt.hour
df['order_hour'] = ordered_hour.fillna(picked_hour)

df['day_of_week'] = df['order_date'].dt.day_name()
df['is_weekend']  = df['day_of_week'].isin(['Saturday', 'Sunday'])
df['month']       = df['order_date'].dt.month_name()

df['time_slot'] = pd.cut(df['order_hour'], bins=[0, 11, 16, 21, 24],
                         labels=['Morning', 'Afternoon', 'Evening', 'Night'], right=False)

# Rider city code from the rider ID, e.g. 'BANGRES18DEL02' -> 'BANG'
df['city_code'] = df['delivery_person_id'].str.extract(r'^([A-Z]+?)RES')

# Late delivery = slower than 75% of all orders (justify this threshold in your README)
late_cutoff = df['time_taken_min'].quantile(0.75)
df['is_late'] = df['time_taken_min'] > late_cutoff
print("Late cutoff (min):", late_cutoff, "| share of late orders:", df['is_late'].mean().round(3))

## 6. Handle remaining missing values

| Column | Decision | Reason |
|---|---|---|
| age, ratings | Fill with **median** | Numeric, and the median is not pulled by extreme values |
| weather, traffic, festival, city | Fill with **"Unknown"** | Don't guess categories; keep the rows and show the gap |
| multiple_deliveries | Fill with **mode** | Discrete count, most orders carry 1 |
| restaurant coordinates, distance | **Leave as NaN** | Can't be guessed; exclude these rows only when analysing distance |

In [ ]:
df['delivery_person_age']     = df['delivery_person_age'].fillna(df['delivery_person_age'].median())
df['delivery_person_ratings'] = df['delivery_person_ratings'].fillna(df['delivery_person_ratings'].median())
df['multiple_deliveries']     = df['multiple_deliveries'].fillna(df['multiple_deliveries'].mode()[0])

for c in ['weatherconditions', 'road_traffic_density', 'festival', 'city']:
    df[c] = df[c].fillna('Unknown')

df.isna().sum()[df.isna().sum() > 0]

## 7. Final checks and save

In [ ]:
print(df.shape)
print("Duplicates:", df.duplicated().sum())
df.describe().round(2)

In [ ]:
df.to_csv("../data/clean/train_clean.csv", index=False)
print("Saved to data/clean/train_clean.csv")